# Image Captioning (BLIP)

Generates a natural-language caption for each survey image using `Salesforce/blip-image-captioning-large` via the HuggingFace Inference API. Adds an `Image_Caption#long` column.

**Requires:** survey images on NFS storage (`has_images` capability). This notebook is only available for surveys where full-size images have been generated.

A free HuggingFace token improves rate limits. Get one at https://huggingface.co/settings/tokens

In [ ]:
# ── Colab bootstrap (no-op on Binder / JupyterHub) ────────────────────────
import os, sys
if "google.colab" in sys.modules:
    if not os.path.isfile("/tmp/suave-nb/helpers/suave_utils.py"):
        import subprocess
        _r = subprocess.run(
            ["git", "clone", "--depth=1", "https://github.com/izaslavsky/suave-notebooks.git", "/tmp/suave-nb"],
            capture_output=True, text=True
        )
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb first.')
else:
    # Colab: load from Drive automatically; prompt only if Drive has no session
    _p = su.load_params(_silent=True)
    if _p is None:
        from IPython.display import display, HTML
        import getpass
        display(HTML('<p style="color:#e07000">No session found on Drive. '
                     'Enter credentials from SuAVEDispatch:</p>'))
        _token = getpass.getpass('SUAVE_TOKEN: ')
        _host  = input('SUAVE_HOST (e.g. george.tirebiter.org): ')
        _p = su.load_params(token=_token, host=_host)

user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

if not _caps.get('has_images'):
    raise RuntimeError(
        'Full-size images are not available for this survey. '
        'Image captioning requires NFS-mounted images.'
    )

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np

In [ ]:
# Optional HuggingFace token — improves rate limits, not required
# Get a free token at https://huggingface.co/settings/tokens
HF_TOKEN = ''   # @param {type:"string"}

client = su.get_hf_client(HF_TOKEN)

## 1. Load data and identify image column

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

# Find the #img column (contains image filenames)
img_cols = [c for c in df.columns if '#img' in c]
if not img_cols:
    raise ValueError('No #img column found in this survey.')
img_col = img_cols[0]
print(f"Image column: {img_col}")

# Build image URLs from the full_images directory
origin = _urlparse(survey_url)
host   = f"{origin.scheme}://{origin.netloc}"

## 2. Generate captions

In [ ]:
MODEL = 'Salesforce/blip-image-captioning-large'
from tqdm.auto import tqdm

captions = []
for img_name in tqdm(df[img_col], desc='Captioning'):
    if not img_name or pd.isna(img_name):
        captions.append(None)
        continue
    # Construct image URL from NFS path converted to URL
    img_path = os.path.join(full_images, str(img_name))
    if not os.path.isfile(img_path):
        captions.append(None)
        continue
    try:
        with open(img_path, 'rb') as f:
            img_bytes = f.read()
        result = client.image_to_text(img_bytes, model=MODEL)
        captions.append(result.generated_text.strip() if hasattr(result, 'generated_text') else str(result).strip())
    except Exception as e:
        captions.append(None)

df['Image_Caption#long'] = captions
n_ok = sum(1 for c in captions if c)
printmd(f"**Generated captions for {n_ok}/{len(df)} images.**")
display(df[[img_col, 'Image_Caption#long']].dropna().head(5))

## 3. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
#Input survey name
survey_name = input('Enter survey name: ')
printmd('**Survey Name:** ' + survey_name)
